# 10 · PyTorch MLP
Architecture: standardised gene-feature vector → [Linear → LayerNorm → GELU → Dropout] × L → Linear (n_genes). Loss: MSE on delta scaled by its global std. Early stopping uses **held-out training perturbations** (never test perturbations). Hyperparameters are fixed in `configs/model_config.yaml`; if you tune, do it *inside* each training split.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import torch
from perturbai.pipeline import load_task
from perturbai.models.mlp import MLPModel
from perturbai.models.baselines import MeanBaseline, RidgeModel
from perturbai.evaluation.benchmark import run_benchmark, summarize, compare_to_baseline
from perturbai.evaluation.splits import split_perturbations
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
T = load_task(P, TAG, ("coexp_",)); H = MCFG["models"]["mlp"]
if SYNTHETIC: H = {**H, "epochs": 80, "hidden": [128, 128]}
mk = lambda: MLPModel(hidden=H["hidden"], dropout=H["dropout"], lr=H["lr"], weight_decay=H["weight_decay"],
                      epochs=H["epochs"], batch_size=H["batch_size"], patience=H["patience"])

In [ ]:
# Training curve on one split
sp0 = split_perturbations(T["index"], 0.2, seed=0); pos = {p: i for i, p in enumerate(T["index"])}
tr = [pos[p] for p in sp0["train"]]
m = mk().fit(T["X"][tr], T["Y"][tr])
h = pd.DataFrame(m.history); h.plot(x="epoch", y=["train_loss", "val_loss"]); plt.yscale("log"); plt.title("MLP training"); plt.show()

In [ ]:
long = run_benchmark({"mlp": mk}, T["X"], T["Y"], T["index"], T["ctrl_mean"], seeds=MCFG["split"]["seeds"],
                     test_frac=MCFG["split"]["test_frac"], top_k=MCFG["metrics"]["top_k"])
long.to_csv(P.results / f"{TAG}_bench_mlp.csv", index=False)
base = pd.read_csv(P.results / f"{TAG}_bench_baselines.csv")
allr = pd.concat([base, long]); display(summarize(allr).round(3)); display(compare_to_baseline(allr, "mean").round(3))

**Decision rule:** the MLP earns a place in the final product only if it beats Ridge/kNN with a margin larger than the seed-to-seed variation. Write your verdict here, even if the answer is "no".